# LegisAnalytica — Exportação Gold → Supabase (PostgreSQL)

**Recorte temporal: 2025-01-01 a 2025-12-31** (um ano-calendário completo)

Notebook de transferência das **26 tabelas Gold** (`workspace.pi_ii_gold`) para o **PostgreSQL do Supabase**.

## Requisitos

- **Secret Scope** `legisanalytica_supabase` com chaves: `jdbc_url`, `user`, `password`
- **Driver** `psycopg` v3 com implementação Python, validada antes da carga
- **Migration SQL** `001_initial_schema_supabase.sql` aplicada previamente no Supabase
- Nenhuma credencial é exposta no código, logs ou GitHub

## Estratégia

1. Lê as 26 tabelas Gold do Unity Catalog via Spark (fatos de 2025 e entidades relacionadas)
2. Transfere em lotes (batch) respeitando a ordem de dependência das FKs
3. Usa `INSERT ... ON CONFLICT DO NOTHING` para evitar duplicidades (não atualiza linhas existentes)
4. Valida contagens de linhas e integridade referencial ao final

## Ordem de execução

| Passo | Descrição |
|-------|-----------|
| 1 | Verificar o driver `psycopg` |
| 2 | Configurar tabelas e ordem de FK |
| 3 | Conectar ao Supabase via Secret Scope |
| 4 | Escolher `EXPORT_MODE`: prepare, pilot ou full |
| 5 | Validar contagens e integridade |
| 6 | Monitorar banco e WAL; acompanhar também o disco físico pelo painel do Supabase |

---


In [0]:
%pip install psycopg

In [0]:
# ============================================================
# psycopg v3. Forçamos a implementação Python antes do import
# porque a versão binária já provocou travamento do kernel neste
# ambiente. Requer libpq disponível no runtime.
# Instalação, se necessária, numa célula separada:
# %pip install psycopg
# Após instalar, iniciar uma nova sessão Python.
import os
os.environ["PSYCOPG_IMPL"] = "python"

try:
    import psycopg
except ImportError as exc:
    raise ImportError(
        "Instale 'psycopg' em uma célula separada e reinicie a sessão Python."
    ) from exc

print(f"psycopg {psycopg.__version__}; implementação: {psycopg.pq.__impl__}")

In [0]:
# ============================================================
# Passo 2 — Configuração e recorte temporal
# ============================================================
# Definição das 26 tabelas Gold, suas colunas, PKs, ordem de
# dependência de FKs (ordem topológica) e filtro temporal.
# As credenciais são lidas do Secret Scope — nunca hardcodeadas.

import re
from datetime import datetime as _dt

# --- Secret Scope ---
SECRET_SCOPE = "legisanalytica_supabase"

# --- Schema Gold no Unity Catalog ---
GOLD_SCHEMA = "workspace.pi_ii_gold"

# --- Tamanho do lote (batch) para inserção ---
BATCH_SIZE = 5000

# --- Recorte temporal (parametrizado) ---
DATA_INICIO = "2025-01-01"
DATA_FIM = "2025-12-31"
ANO_INICIO = _dt.strptime(DATA_INICIO, "%Y-%m-%d").year  # 2025
ANO_FIM = _dt.strptime(DATA_FIM, "%Y-%m-%d").year        # 2025

# --- Limites de armazenamento Supabase Free ---
# A execução começa SEM gravações. Alterar manualmente após revisar:
#   prepare: consulta de conexão/definições; nenhuma inserção
#   pilot: exporta somente a tabela CASA (2 registros)
#   full: exporta as 26 tabelas, com validação final
EXPORT_MODE = "prepare"

PG_SIZE_WARN_MB = 300       # inclui tabelas, índices e objetos do banco
PG_SIZE_ABORT_MB = 350      # margem operacional antes dos 500 MB
WAL_SIZE_WARN_MB = 1200
WAL_SIZE_ABORT_MB = 1400
# Conservador com base na sobra de disco observada no projeto anterior.
# Não substitui a medição do disco físico no painel do Supabase.
DB_PLUS_WAL_ABORT_MB = 1300

if EXPORT_MODE not in {"prepare", "pilot", "full"}:
    raise ValueError("EXPORT_MODE deve ser prepare, pilot ou full")

# --- Configuração das 26 tabelas (ordem topológica de FK) ---
# Cada entrada: (gold_table, colunas, pk_colunas, filter_sql)
# filter_sql = None  → exportar tabela completa (catálogos, dims)
# filter_sql = string → ler subconjunto via spark.sql(filter_sql)

_G = GOLD_SCHEMA  # alias para legibilidade

# Proposições do ano, mais as anteriores referenciadas por votações de 2025.
# Essas proposições mais antigas são apenas pais necessários à integridade;
# NÃO constituem proposições apresentadas em 2025 para indicadores anuais.
_PROPOSICOES_2025_SQL = f"""
SELECT p.* FROM {_G}.proposicao p
WHERE (p.data_apresentacao BETWEEN DATE '{DATA_INICIO}' AND DATE '{DATA_FIM}')
   OR (p.data_apresentacao IS NULL AND p.ano = {ANO_INICIO})
   OR EXISTS (
       SELECT 1 FROM {_G}.votacao_proposicao vp
       JOIN {_G}.votacao v
         ON v.casa = vp.casa AND v.id_votacao = vp.id_votacao
       WHERE vp.casa = p.casa AND vp.id_proposicao = p.id_proposicao
         AND v.data_votacao BETWEEN DATE '{DATA_INICIO}' AND DATE '{DATA_FIM}'
   )
"""

TABLES = [
    # NÍVEL 0 — Catálogos raiz (COMPLETOS)
    ("lote",
     ["id_lote", "entidade_fonte", "localizador", "hash_sha256",
      "processado_em", "coletado_em", "cobertura_inicio", "cobertura_fim"],
     ["id_lote"], None),
    ("casa", ["casa", "nome"], ["casa"], None),
    ("uf", ["uf", "nome"], ["uf"], None),
    ("legislatura", ["numero_legislatura", "inicio", "fim"], ["numero_legislatura"], None),
    ("partido", ["sigla"], ["sigla"], None),
    ("tipo_proposicao", ["sigla_tipo", "descricao"], ["sigla_tipo"], None),

    # NÍVEL 1 (COMPLETOS)
    ("causa_afastamento", ["casa", "codigo_causa", "descricao"], ["casa", "codigo_causa"], None),
    ("parlamentar",
     ["casa", "id_parlamentar", "nome_referencia", "url_foto", "email", "id_lote"],
     ["casa", "id_parlamentar"], None),
    ("tipo_voto", ["casa", "codigo_voto", "descricao"], ["casa", "codigo_voto"], None),
    ("tipo_despesa", ["casa", "codigo_tipo", "descricao"], ["casa", "codigo_tipo"], None),

    # NÍVEL 2
    ("identificador_parlamentar",
     ["casa", "sistema", "codigo_externo", "id_parlamentar"],
     ["casa", "sistema", "codigo_externo"], None),
    ("cadastro_observado",
     ["casa", "id_parlamentar", "id_lote", "partido", "uf"],
     ["casa", "id_parlamentar", "id_lote"], None),
    ("parlamentar_legislatura",
     ["casa", "id_parlamentar", "numero_legislatura", "criterio_inclusao", "id_lote"],
     ["casa", "id_parlamentar", "numero_legislatura"], None),
    ("mandato", ["casa", "id_mandato", "uf", "id_lote"], ["casa", "id_mandato"], None),
    # fornecedor: apenas os referenciados por despesas do recorte
    ("fornecedor", ["casa", "cnpj_cpf", "nome", "id_lote"], ["casa", "cnpj_cpf"],
     f"""SELECT f.* FROM {_G}.fornecedor f
         INNER JOIN (SELECT DISTINCT casa, cnpj_cpf FROM {_G}.despesa
                     WHERE ano >= {ANO_INICIO} AND ano <= {ANO_FIM}
                       AND cnpj_cpf IS NOT NULL) d
         ON f.casa = d.casa AND f.cnpj_cpf = d.cnpj_cpf"""),
    # proposicao: 2025 + pais anteriores necessários às votações de 2025
    ("proposicao",
     ["casa", "id_proposicao", "sigla_tipo", "numero", "ano", "ementa",
      "data_apresentacao", "url_inteiro_teor", "id_lote"],
     ["casa", "id_proposicao"], _PROPOSICOES_2025_SQL),

    # NÍVEL 3
    ("vinculo_mandato",
     ["casa", "id_mandato", "id_parlamentar", "participacao", "id_lote"],
     ["casa", "id_mandato", "id_parlamentar"], None),
    # autoria: INNER JOIN com proposicao do recorte
    ("autoria",
     ["casa", "id_proposicao", "id_autor", "nome_autor", "partido", "uf",
      "ordem_assinatura", "id_lote"],
     ["casa", "id_proposicao", "id_autor"],
     f"""SELECT a.* FROM {_G}.autoria a
         INNER JOIN ({_PROPOSICOES_2025_SQL}) p
           ON a.casa = p.casa AND a.id_proposicao = p.id_proposicao"""),
    # status_atual: INNER JOIN com proposicao do recorte
    ("status_atual",
     ["casa", "id_proposicao", "data_hora", "sigla_orgao",
      "descricao_tramitacao", "descricao_situacao", "apreciacao", "id_lote"],
     ["casa", "id_proposicao"],
     f"""SELECT s.* FROM {_G}.status_atual s
         INNER JOIN ({_PROPOSICOES_2025_SQL}) p
           ON s.casa = p.casa AND s.id_proposicao = p.id_proposicao"""),
    # votacao: filtrar por data_votacao
    ("votacao",
     ["casa", "id_votacao", "data_votacao", "sigla_orgao", "aprovacao",
      "descricao", "votos_sim", "votos_nao", "votos_outros", "id_lote"],
     ["casa", "id_votacao"],
     f"SELECT * FROM {_G}.votacao WHERE data_votacao >= '{DATA_INICIO}' AND data_votacao <= '{DATA_FIM}'"),
    # despesa: filtrar por ano
    ("despesa",
     ["casa", "id_despesa", "id_parlamentar", "ano", "mes", "data_emissao",
      "codigo_tipo", "cnpj_cpf", "id_lote"],
     ["casa", "id_despesa"],
     f"SELECT * FROM {_G}.despesa WHERE ano >= {ANO_INICIO} AND ano <= {ANO_FIM}"),
    # total_publicado: filtrar por ano
    ("total_publicado", ["casa", "ano", "total_proposicoes", "id_lote"], ["casa", "ano"],
     f"SELECT * FROM {_G}.total_publicado WHERE ano >= {ANO_INICIO} AND ano <= {ANO_FIM}"),

    # NÍVEL 4
    ("exercicio",
     ["casa", "id_exercicio", "id_mandato", "id_parlamentar", "inicio", "fim",
      "codigo_causa", "id_lote"],
     ["casa", "id_exercicio"], None),
    # votacao_proposicao: TODAS as ligações das votações de 2025;
    # proposições relacionadas fora do ano são incluídas na seleção acima.
    ("votacao_proposicao",
     ["casa", "id_votacao", "id_proposicao", "id_lote"],
     ["casa", "id_votacao", "id_proposicao"],
     f"""SELECT vp.* FROM {_G}.votacao_proposicao vp
         INNER JOIN {_G}.votacao v
           ON vp.casa = v.casa AND vp.id_votacao = v.id_votacao
         WHERE v.data_votacao BETWEEN DATE '{DATA_INICIO}' AND DATE '{DATA_FIM}'"""),
    # valor_despesa: INNER JOIN com despesa do recorte
    ("valor_despesa",
     ["casa", "id_despesa", "valor_documento", "valor_glosa", "valor_liquido", "id_lote"],
     ["casa", "id_despesa"],
     f"""SELECT vd.* FROM {_G}.valor_despesa vd
         INNER JOIN {_G}.despesa d
           ON vd.casa = d.casa AND vd.id_despesa = d.id_despesa
         WHERE d.ano >= {ANO_INICIO} AND d.ano <= {ANO_FIM}"""),

    # NÍVEL 5
    # voto: INNER JOIN com votacao do recorte
    ("voto",
     ["casa", "id_votacao", "id_parlamentar", "codigo_voto", "id_lote"],
     ["casa", "id_votacao", "id_parlamentar"],
     f"""SELECT vt.* FROM {_G}.voto vt
         INNER JOIN {_G}.votacao v
           ON vt.casa = v.casa AND vt.id_votacao = v.id_votacao
         WHERE v.data_votacao >= '{DATA_INICIO}' AND v.data_votacao <= '{DATA_FIM}'"""),
]

print(f"Configuradas {len(TABLES)} tabelas para exportação.")
print(f"Recorte temporal: {DATA_INICIO} a {DATA_FIM} (anos {ANO_INICIO}-{ANO_FIM})")
print(f"Batch size: {BATCH_SIZE} linhas por lote.")
print(f"Secret Scope: {SECRET_SCOPE}")
print(f"Modo de execução: {EXPORT_MODE} (prepare = sem escrita)")
print(f"Limite do banco: warn={PG_SIZE_WARN_MB}MB, abort={PG_SIZE_ABORT_MB}MB")

In [0]:
# ============================================================
# dbutils é disponibilizado automaticamente no notebook Databricks.
# Não criar instância de DBUtils: usar o objeto integrado ao notebook.
# A JDBC URL é convertida removendo apenas o prefixo "jdbc:".

if "dbutils" not in globals():
    raise RuntimeError("Execute este notebook no Databricks, com dbutils disponível.")


def get_connection():
    jdbc_url = dbutils.secrets.get(scope=SECRET_SCOPE, key="jdbc_url")
    user = dbutils.secrets.get(scope=SECRET_SCOPE, key="user")
    password = dbutils.secrets.get(scope=SECRET_SCOPE, key="password")

    # Mantém database e query params (inclusive ?sslmode=require).
    uri = jdbc_url.removeprefix("jdbc:")
    if not uri.startswith("postgresql://"):
        raise ValueError("jdbc_url precisa conter uma URL JDBC PostgreSQL válida.")

    return psycopg.connect(
        conninfo=uri,
        user=user,
        password=password,
        sslmode="require",
        connect_timeout=20,
        autocommit=False,
    )


print("Testando conexão com Supabase...")
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT current_database(), current_user")
        banco, usuario = cur.fetchone()
print(f"Conexão OK. Banco: {banco}; usuário: {usuario}")

In [0]:
# ============================================================
# O PostgreSQL mantém WAL independente do tamanho das tabelas.
# O total (banco + WAL) NÃO equivale à ocupação física total:
# acompanhar a métrica Disk no painel do Supabase durante a carga.

import time


def build_insert_sql(table_name, columns, pk_columns):
    # Nomes de tabelas e colunas vêm apenas da configuração estática TABLES.
    cols = ", ".join(columns)
    placeholders = ", ".join(["%s"] * len(columns))
    pk_cols = ", ".join(pk_columns)
    return (f"INSERT INTO public.{table_name} ({cols}) "
            f"VALUES ({placeholders}) ON CONFLICT ({pk_cols}) DO NOTHING")


def scalar(conn, sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchone()[0]


def storage_metrics(conn):
    # Falha de medição interrompe a importação (não ignora o erro).
    # float(): pg_ls_waldir devolve numeric (Decimal) no psycopg e
    # Decimal + float levantaria TypeError na soma.
    db_mb = float(scalar(conn, "SELECT pg_database_size(current_database())")) / 1048576
    wal_mb = float(scalar(conn, "SELECT COALESCE(SUM(size), 0) FROM pg_ls_waldir()")) / 1048576
    return db_mb, wal_mb


def check_storage(conn):
    db_mb, wal_mb = storage_metrics(conn)
    combined = db_mb + wal_mb
    print(f"    Armazenamento: banco={db_mb:.1f} MB, WAL={wal_mb:.1f} MB")
    if (db_mb >= PG_SIZE_ABORT_MB or wal_mb >= WAL_SIZE_ABORT_MB
            or combined >= DB_PLUS_WAL_ABORT_MB):
        raise RuntimeError(
            "Limiar de segurança de armazenamento alcançado. "
            "Interrompa e confira o disco físico no painel Supabase."
        )
    if db_mb >= PG_SIZE_WARN_MB or wal_mb >= WAL_SIZE_WARN_MB:
        print("    ATENÇÃO: armazenamento próximo dos limites definidos.")
    return db_mb, wal_mb


def export_table(gold_table, columns, pk_columns, conn, filter_sql=None,
                 batch_size=BATCH_SIZE):
    start = time.time()
    df = spark.sql(filter_sql) if filter_sql else spark.table(f"{GOLD_SCHEMA}.{gold_table}")
    gold_count = df.count()
    pg_before = scalar(conn, f"SELECT COUNT(*) FROM public.{gold_table}")
    if not gold_count:
        print(f"  {gold_table}: fonte vazia; banco já contém {pg_before} registros.")
        return dict(table=gold_table, gold_count=0, pg_inserted=0,
                    pg_final=pg_before, elapsed_s=0.0, status="source_empty")

    check_storage(conn)
    insert_sql = build_insert_sql(gold_table, columns, pk_columns)
    inserted_total = 0
    batch_number = 0
    batch = []

    def write_batch(rows):
        nonlocal inserted_total, batch_number
        check_storage(conn)
        batch_number += 1
        try:
            with conn.cursor() as cur:
                cur.executemany(insert_sql, rows)
                inserted_total += max(0, cur.rowcount)
            conn.commit()
            # Verificação APÓS commit; interrompe antes do próximo lote.
            check_storage(conn)
        except Exception as exc:
            conn.rollback()
            raise RuntimeError(
                f"Falha na tabela {gold_table}, lote {batch_number}. "
                "Importação interrompida; lotes anteriores permanecem no banco."
            ) from exc

    # Sem repartition/shuffle adicional: streaming por partição de origem.
    for row in df.select(*columns).toLocalIterator():
        batch.append(tuple(row[col] for col in columns))
        if len(batch) == batch_size:
            write_batch(batch)
            batch = []
            if batch_number % 10 == 0:
                print(f"    {gold_table}: {batch_number} lotes processados")
    if batch:
        write_batch(batch)

    pg_after = scalar(conn, f"SELECT COUNT(*) FROM public.{gold_table}")
    status = "ok" if pg_after == gold_count else "count_mismatch"
    print(f"  {gold_table}: Gold={gold_count:,}; PostgreSQL={pg_after:,}; "
          f"inseridos={inserted_total:,}; status={status}")
    if status != "ok":
        raise RuntimeError(
            f"Contagens divergentes em {gold_table}: fonte={gold_count}, destino={pg_after}. "
            "A execução é idempotente, mas não corrige diferenças em registros existentes."
        )
    return dict(table=gold_table, gold_count=gold_count,
                pg_inserted=inserted_total, pg_final=pg_after,
                elapsed_s=round(time.time() - start, 1), status=status)

In [0]:
# ============================================================
# Passo 3.5 — Aplicar migration SQL no Supabase (se ainda não aplicada)
# ============================================================
# Lê o arquivo 001_initial_schema_supabase.sql do workspace e executa
# no PostgreSQL do Supabase. Idempotente: verifica se a tabela 'lote'
# já existe antes de tentar criar.

import pathlib

MIGRATION_PATH = "/Workspace/Users/brucewellington@hotmail.com/PI_II_Univesp/03_Gold/supabase/migrations/001_initial_schema_supabase.sql"
migration_sql = pathlib.Path(MIGRATION_PATH).read_text()

print(f"Migration SQL carregada: {len(migration_sql):,} caracteres")

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT EXISTS (
                SELECT 1 FROM information_schema.tables
                WHERE table_schema = 'public' AND table_name = 'lote'
            )
        """)
        already_exists = cur.fetchone()[0]

    if already_exists:
        print("Schema já existe no Supabase — migration NÃO executada.")
    else:
        print("Schema não encontrado — executando migration...")
        with conn.cursor() as cur:
            cur.execute(migration_sql)
        conn.commit()
        print("Migration aplicada com sucesso!")
        with conn.cursor() as cur:
            cur.execute("""
                SELECT COUNT(*) FROM information_schema.tables
                WHERE table_schema = 'public'
            """)
            n_tables = cur.fetchone()[0]
        print(f"Tabelas criadas no schema public: {n_tables}")

In [0]:
# ============================================================
# Passo 3.7 — Pre-flight: recorte × constraints da migration
# ============================================================
# A migration 001_initial_schema_supabase.sql (já aplicada no
# Supabase) impõe NOT NULL, UNIQUE (lote.entidade_fonte,
# lote.hash_sha256), VARCHAR(n) e PKs. Uma violação interromperia
# a carga no meio de uma tabela. Este passo valida tudo ANTES da
# primeira escrita — no Spark (dados) e no PostgreSQL (schema):
#   1. NULLs em colunas NOT NULL da migration, por subset exportado
#   2. Duplicidades de PK no subset (evita perda silenciosa no
#      ON CONFLICT DO NOTHING)
#   3. UNIQUE de lote (entidade_fonte, hash_sha256) e CHAR(64)
#   4. Limites VARCHAR: casa(10), uf(2), sistema(10)
#   5. Schema public: 26 tabelas com as colunas esperadas
# FKs não são checadas aqui: a estratégia FK-safe (INNER JOIN com
# pais do recorte) garante zero órfãos por construção, e o Passo 7
# revalida as 44 FKs no PostgreSQL após a carga.
# Qualquer falha aborta o notebook sem escrever nada.

from pyspark.sql import functions as F

# Colunas NOT NULL por tabela, conforme a migration aplicada
NOT_NULL = {
    "lote": ["id_lote", "entidade_fonte", "localizador", "hash_sha256", "processado_em"],
    "casa": ["casa", "nome"],
    "uf": ["uf", "nome"],
    "legislatura": ["numero_legislatura", "inicio", "fim"],
    "partido": ["sigla"],
    "tipo_proposicao": ["sigla_tipo"],
    "causa_afastamento": ["casa", "codigo_causa", "descricao"],
    "parlamentar": ["casa", "id_parlamentar", "nome_referencia", "id_lote"],
    "tipo_voto": ["casa", "codigo_voto"],
    "tipo_despesa": ["casa", "codigo_tipo", "descricao"],
    "identificador_parlamentar": ["casa", "sistema", "codigo_externo"],
    "cadastro_observado": ["casa", "id_parlamentar", "id_lote"],
    "parlamentar_legislatura": ["casa", "id_parlamentar", "numero_legislatura", "criterio_inclusao", "id_lote"],
    "mandato": ["casa", "id_mandato", "uf", "id_lote"],
    "fornecedor": ["casa", "cnpj_cpf", "nome", "id_lote"],
    "proposicao": ["casa", "id_proposicao", "sigla_tipo", "id_lote"],
    "vinculo_mandato": ["casa", "id_mandato", "id_parlamentar", "participacao", "id_lote"],
    "autoria": ["casa", "id_proposicao", "id_autor", "id_lote"],
    "status_atual": ["casa", "id_proposicao", "id_lote"],
    "votacao": ["casa", "id_votacao", "id_lote"],
    "despesa": ["casa", "id_despesa", "ano", "mes", "id_lote"],
    "total_publicado": ["casa", "ano", "total_proposicoes", "id_lote"],
    "exercicio": ["casa", "id_exercicio", "id_mandato", "id_parlamentar", "inicio", "id_lote"],
    "votacao_proposicao": ["casa", "id_votacao", "id_proposicao", "id_lote"],
    "valor_despesa": ["casa", "id_despesa", "id_lote"],
    "voto": ["casa", "id_votacao", "id_parlamentar", "id_lote"],
}

# Subsets idênticos aos exportados (derivados de TABLES, sem duplicar filtros)
SUBSETS = {t: (spark.sql(f) if f else spark.table(f"{GOLD_SCHEMA}.{t}"))
           for t, _, _, f in TABLES}

preflight_errors = []

# 1) NOT NULL da migration, por subset
for t, cols in NOT_NULL.items():
    row = SUBSETS[t].agg(
        *[F.sum(F.col(c).isNull().cast("long")).alias(c) for c in cols]
    ).first()
    for c in cols:
        if row[c]:
            preflight_errors.append(
                f"{t}.{c}: {int(row[c])} NULL(s) — viola NOT NULL da migration")

# 2) Duplicidade de PK nos subsets
for gold_table, columns, pk_columns, _ in TABLES:
    df = SUBSETS[gold_table]
    n, n_pk = df.count(), df.select(*pk_columns).distinct().count()
    if n != n_pk:
        preflight_errors.append(
            f"{gold_table}: {n - n_pk} duplicidade(s) de PK no subset")

# 3) lote — UNIQUE (entidade_fonte), UNIQUE (hash_sha256), CHAR(64)
_lote = SUBSETS["lote"]
_n_lote = _lote.count()
for _c in ("entidade_fonte", "hash_sha256"):
    if _lote.select(_c).distinct().count() != _n_lote:
        preflight_errors.append(f"lote.{_c}: duplicidade viola UNIQUE da migration")
_n_long = _lote.where(F.length("hash_sha256") > 64).count()
if _n_long:
    preflight_errors.append(f"lote.hash_sha256: {_n_long} valor(es) acima de CHAR(64)")

# 4) Limites VARCHAR da migration
for t, c, lim in (("parlamentar", "casa", 10), ("mandato", "uf", 2),
                  ("identificador_parlamentar", "sistema", 10)):
    _n = SUBSETS[t].where(F.length(c) > lim).count()
    if _n:
        preflight_errors.append(f"{t}.{c}: {_n} valor(es) excedem VARCHAR({lim})")

# 5) Schema public do Supabase: tabelas e colunas esperadas
with get_connection() as conn:
    for gold_table, columns, pk_columns, _ in TABLES:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT column_name FROM information_schema.columns "
                "WHERE table_schema = 'public' AND table_name = %s",
                (gold_table,),
            )
            pg_cols = {r[0] for r in cur.fetchall()}
        if not pg_cols:
            preflight_errors.append(f"public.{gold_table}: tabela não encontrada no Supabase")
            continue
        for c in sorted(set(columns) - pg_cols):
            preflight_errors.append(f"public.{gold_table}.{c}: coluna ausente no PostgreSQL")
        for c in sorted(pg_cols - set(columns)):
            preflight_errors.append(f"public.{gold_table}.{c}: coluna extra no PostgreSQL")

print(f"Pre-flight: {len(SUBSETS)} subsets validados; "
      f"{len(preflight_errors)} erro(s) encontrado(s).")
if preflight_errors:
    for e in preflight_errors:
        print("  FALHA:", e)
    raise RuntimeError("Pre-flight reprovado — a exportação NÃO foi iniciada.")
print("Pre-flight APROVADO: NOT NULL, PKs, UNIQUE, VARCHAR e schema do PostgreSQL verificados.")

In [0]:
# ============================================================
# A escrita está DESLIGADA por padrão.
# prepare: só prepara o notebook; pilot: exporta public.casa;
# full: exporta as 26 tabelas e habilita a validação integral.

results = []
export_completed = False

if EXPORT_MODE == "prepare":
    print("Modo prepare: NENHUMA escrita executada no Supabase.")
else:
    selected = [t for t in TABLES if t[0] == "casa"] if EXPORT_MODE == "pilot" else TABLES
    print(f"Iniciando {EXPORT_MODE}: {len(selected)} tabela(s), período {DATA_INICIO} a {DATA_FIM}.")
    with get_connection() as conn:
        try:
            for i, (gold_table, columns, pk_columns, filter_sql) in enumerate(selected, 1):
                print(f"[{i}/{len(selected)}] {gold_table}")
                results.append(export_table(gold_table, columns, pk_columns, conn, filter_sql))
        except BaseException:
            conn.rollback()
            raise
    export_completed = True
    print(f"Execução {EXPORT_MODE} concluída: {len(results)} tabela(s).")

In [0]:
# ============================================================
validation_results = []
all_match = False

if EXPORT_MODE != "full" or not export_completed:
    print("Validação das 26 tabelas adiada: disponível após modo full concluído.")
else:
    print("VALIDAÇÃO: contagens Gold filtrada vs PostgreSQL")
    with get_connection() as conn:
        for gold_table, columns, pk_columns, filter_sql in TABLES:
            gold_count = (spark.sql(filter_sql).count() if filter_sql
                          else spark.table(f"{GOLD_SCHEMA}.{gold_table}").count())
            pg_count = scalar(conn, f"SELECT COUNT(*) FROM public.{gold_table}")
            matched = gold_count == pg_count
            validation_results.append(dict(table=gold_table, gold=gold_count,
                                           pg=pg_count, match=matched))
            print(f"  {gold_table}: Gold={gold_count:,}, PG={pg_count:,} "
                  f"{'OK' if matched else 'DIVERGÊNCIA'}")
    all_match = all(r["match"] for r in validation_results)

In [0]:
# ============================================================
# Passo 7 — Validar integridade referencial (FKs)
# ============================================================
# Verifica todas as FKs no PostgreSQL, contando registros órfãos.
# FKs com colunas NULLABLE usam a condição WHERE col IS NOT NULL
# para ignorar NULLs (comportamento MATCH SIMPLE).
# As tabelas filhas foram filtradas por INNER JOIN com os pais no
# recorte 2025, garantindo zero órfãos por construção.

# Lista de verificações: (descrição, SQL de contagem de órfãos)
FK_CHECKS = [
    # NÍVEL 1 → catálogos
    ("causa_afastamento → casa",
     "SELECT COUNT(*) FROM causa_afastamento c LEFT JOIN casa ON c.casa = casa.casa WHERE casa.casa IS NULL"),
    ("parlamentar → casa",
     "SELECT COUNT(*) FROM parlamentar p LEFT JOIN casa ON p.casa = casa.casa WHERE casa.casa IS NULL"),
    ("parlamentar → lote",
     "SELECT COUNT(*) FROM parlamentar p LEFT JOIN lote ON p.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("tipo_voto → casa",
     "SELECT COUNT(*) FROM tipo_voto t LEFT JOIN casa ON t.casa = casa.casa WHERE casa.casa IS NULL"),
    ("tipo_despesa → casa",
     "SELECT COUNT(*) FROM tipo_despesa t LEFT JOIN casa ON t.casa = casa.casa WHERE casa.casa IS NULL"),

    # NÍVEL 2
    ("identificador_parlamentar → parlamentar (NULL ok)",
     "SELECT COUNT(*) FROM identificador_parlamentar i LEFT JOIN parlamentar p ON i.casa = p.casa AND i.id_parlamentar = p.id_parlamentar WHERE i.id_parlamentar IS NOT NULL AND p.id_parlamentar IS NULL"),
    ("cadastro_observado → parlamentar",
     "SELECT COUNT(*) FROM cadastro_observado c LEFT JOIN parlamentar p ON c.casa = p.casa AND c.id_parlamentar = p.id_parlamentar WHERE p.id_parlamentar IS NULL"),
    ("cadastro_observado → lote",
     "SELECT COUNT(*) FROM cadastro_observado c LEFT JOIN lote ON c.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("cadastro_observado → partido (NULL ok)",
     "SELECT COUNT(*) FROM cadastro_observado c LEFT JOIN partido ON c.partido = partido.sigla WHERE c.partido IS NOT NULL AND partido.sigla IS NULL"),
    ("cadastro_observado → uf (NULL ok)",
     "SELECT COUNT(*) FROM cadastro_observado c LEFT JOIN uf ON c.uf = uf.uf WHERE c.uf IS NOT NULL AND uf.uf IS NULL"),
    ("parlamentar_legislatura → parlamentar",
     "SELECT COUNT(*) FROM parlamentar_legislatura pl LEFT JOIN parlamentar p ON pl.casa = p.casa AND pl.id_parlamentar = p.id_parlamentar WHERE p.id_parlamentar IS NULL"),
    ("parlamentar_legislatura → legislatura",
     "SELECT COUNT(*) FROM parlamentar_legislatura pl LEFT JOIN legislatura ON pl.numero_legislatura = legislatura.numero_legislatura WHERE legislatura.numero_legislatura IS NULL"),
    ("parlamentar_legislatura → lote",
     "SELECT COUNT(*) FROM parlamentar_legislatura pl LEFT JOIN lote ON pl.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("mandato → casa",
     "SELECT COUNT(*) FROM mandato m LEFT JOIN casa ON m.casa = casa.casa WHERE casa.casa IS NULL"),
    ("mandato → uf",
     "SELECT COUNT(*) FROM mandato m LEFT JOIN uf ON m.uf = uf.uf WHERE uf.uf IS NULL"),
    ("mandato → lote",
     "SELECT COUNT(*) FROM mandato m LEFT JOIN lote ON m.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("fornecedor → casa",
     "SELECT COUNT(*) FROM fornecedor f LEFT JOIN casa ON f.casa = casa.casa WHERE casa.casa IS NULL"),
    ("fornecedor → lote",
     "SELECT COUNT(*) FROM fornecedor f LEFT JOIN lote ON f.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("proposicao → tipo_proposicao",
     "SELECT COUNT(*) FROM proposicao p LEFT JOIN tipo_proposicao ON p.sigla_tipo = tipo_proposicao.sigla_tipo WHERE tipo_proposicao.sigla_tipo IS NULL"),
    ("proposicao → lote",
     "SELECT COUNT(*) FROM proposicao p LEFT JOIN lote ON p.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),

    # NÍVEL 3
    ("vinculo_mandato → mandato",
     "SELECT COUNT(*) FROM vinculo_mandato v LEFT JOIN mandato m ON v.casa = m.casa AND v.id_mandato = m.id_mandato WHERE m.id_mandato IS NULL"),
    ("vinculo_mandato → parlamentar",
     "SELECT COUNT(*) FROM vinculo_mandato v LEFT JOIN parlamentar p ON v.casa = p.casa AND v.id_parlamentar = p.id_parlamentar WHERE p.id_parlamentar IS NULL"),
    ("vinculo_mandato → lote",
     "SELECT COUNT(*) FROM vinculo_mandato v LEFT JOIN lote ON v.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("autoria → proposicao",
     "SELECT COUNT(*) FROM autoria a LEFT JOIN proposicao p ON a.casa = p.casa AND a.id_proposicao = p.id_proposicao WHERE p.id_proposicao IS NULL"),
    ("autoria → lote",
     "SELECT COUNT(*) FROM autoria a LEFT JOIN lote ON a.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("autoria → partido (NULL ok)",
     "SELECT COUNT(*) FROM autoria a LEFT JOIN partido ON a.partido = partido.sigla WHERE a.partido IS NOT NULL AND partido.sigla IS NULL"),
    ("autoria → uf (NULL ok)",
     "SELECT COUNT(*) FROM autoria a LEFT JOIN uf ON a.uf = uf.uf WHERE a.uf IS NOT NULL AND uf.uf IS NULL"),
    ("status_atual → proposicao",
     "SELECT COUNT(*) FROM status_atual s LEFT JOIN proposicao p ON s.casa = p.casa AND s.id_proposicao = p.id_proposicao WHERE p.id_proposicao IS NULL"),
    ("status_atual → lote",
     "SELECT COUNT(*) FROM status_atual s LEFT JOIN lote ON s.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("votacao → lote",
     "SELECT COUNT(*) FROM votacao v LEFT JOIN lote ON v.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("despesa → tipo_despesa (NULL ok)",
     "SELECT COUNT(*) FROM despesa d LEFT JOIN tipo_despesa t ON d.casa = t.casa AND d.codigo_tipo = t.codigo_tipo WHERE d.codigo_tipo IS NOT NULL AND t.codigo_tipo IS NULL"),
    ("despesa → fornecedor (NULL ok)",
     "SELECT COUNT(*) FROM despesa d LEFT JOIN fornecedor f ON d.casa = f.casa AND d.cnpj_cpf = f.cnpj_cpf WHERE d.cnpj_cpf IS NOT NULL AND f.cnpj_cpf IS NULL"),
    ("despesa → lote",
     "SELECT COUNT(*) FROM despesa d LEFT JOIN lote ON d.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("total_publicado → lote",
     "SELECT COUNT(*) FROM total_publicado t LEFT JOIN lote ON t.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),

    # NÍVEL 4
    ("exercicio → vinculo_mandato",
     "SELECT COUNT(*) FROM exercicio e LEFT JOIN vinculo_mandato v ON e.casa = v.casa AND e.id_mandato = v.id_mandato AND e.id_parlamentar = v.id_parlamentar WHERE v.id_mandato IS NULL"),
    ("exercicio → causa_afastamento (NULL ok)",
     "SELECT COUNT(*) FROM exercicio e LEFT JOIN causa_afastamento c ON e.casa = c.casa AND e.codigo_causa = c.codigo_causa WHERE e.codigo_causa IS NOT NULL AND c.codigo_causa IS NULL"),
    ("exercicio → lote",
     "SELECT COUNT(*) FROM exercicio e LEFT JOIN lote ON e.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("votacao_proposicao → votacao",
     "SELECT COUNT(*) FROM votacao_proposicao vp LEFT JOIN votacao v ON vp.casa = v.casa AND vp.id_votacao = v.id_votacao WHERE v.id_votacao IS NULL"),
    ("votacao_proposicao → proposicao",
     "SELECT COUNT(*) FROM votacao_proposicao vp LEFT JOIN proposicao p ON vp.casa = p.casa AND vp.id_proposicao = p.id_proposicao WHERE p.id_proposicao IS NULL"),
    ("votacao_proposicao → lote",
     "SELECT COUNT(*) FROM votacao_proposicao vp LEFT JOIN lote ON vp.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("valor_despesa → despesa",
     "SELECT COUNT(*) FROM valor_despesa vd LEFT JOIN despesa d ON vd.casa = d.casa AND vd.id_despesa = d.id_despesa WHERE d.id_despesa IS NULL"),
    ("valor_despesa → lote",
     "SELECT COUNT(*) FROM valor_despesa vd LEFT JOIN lote ON vd.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),

    # NÍVEL 5
    ("voto → votacao",
     "SELECT COUNT(*) FROM voto v LEFT JOIN votacao ON v.casa = votacao.casa AND v.id_votacao = votacao.id_votacao WHERE votacao.id_votacao IS NULL"),
    ("voto → parlamentar",
     "SELECT COUNT(*) FROM voto v LEFT JOIN parlamentar p ON v.casa = p.casa AND v.id_parlamentar = p.id_parlamentar WHERE p.id_parlamentar IS NULL"),
    ("voto → tipo_voto (NULL ok)",
     "SELECT COUNT(*) FROM voto v LEFT JOIN tipo_voto t ON v.casa = t.casa AND v.codigo_voto = t.codigo_voto WHERE v.codigo_voto IS NOT NULL AND t.codigo_voto IS NULL"),
    ("voto → lote",
     "SELECT COUNT(*) FROM voto v LEFT JOIN lote ON v.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
]

all_fk_ok = False
if EXPORT_MODE != "full" or not export_completed:
    print("Validação de FKs adiada: disponível após modo full concluído.")
else:
    print(f"VALIDAÇÃO: INTEGRIDADE REFERENCIAL ({len(FK_CHECKS)} consultas)")
    with get_connection() as conn:
        failures = []
        for desc, query in FK_CHECKS:
            orphans = scalar(conn, query)
            if orphans:
                failures.append((desc, orphans))
            print(f"  {desc}: {orphans} órfão(s)")
    all_fk_ok = not failures
    print("FKs OK" if all_fk_ok else f"FALHAS: {failures}")

In [0]:
# ============================================================
print("=" * 60)
print(f"LegisAnalytica → Supabase | modo={EXPORT_MODE} | {DATA_INICIO} a {DATA_FIM}")
print(f"Tabelas processadas: {len(results)}/{len(TABLES)}")

if EXPORT_MODE == "prepare":
    print("PRONTO PARA REVISÃO: exportação não executada.")
elif EXPORT_MODE == "pilot":
    print("PILOTO FINALIZADO: valide public.casa e as métricas Disk no Supabase.")
elif export_completed and all_match and all_fk_ok:
    print("CARGA INTEGRAL VALIDADA: contagens e verificações de FK aprovadas.")
else:
    print("ATENÇÃO: a carga integral não foi validada completamente.")

if export_completed:
    with get_connection() as conn:
        db_mb, wal_mb = storage_metrics(conn)
        print(f"Tamanho banco={db_mb:.1f} MB; WAL={wal_mb:.1f} MB")
    print("Observe também a métrica Disk do projeto no painel do Supabase.")

print("OBS: ON CONFLICT DO NOTHING evita duplicações, mas não atualiza linhas já existentes.")